# Data Exploration

In [12]:
# Load the ratings using the function from data.py
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
from data import load_movielens, split_by_time
from model import build_graph, BehavioralRecommender

In [9]:
df = load_movielens()   # downloads into data/ the first time
df.head(10)

,user,item,rating,time
0,6039,802,4,956703932
1,6039,579,5,956703954
2,6039,2191,4,956703954
3,6039,1781,4,956703977
4,6039,1839,5,956703977
5,6039,559,4,956704056
6,6039,1316,3,956704056
7,6039,207,5,956704056
8,6039,2895,5,956704056
9,6039,3272,4,956704056


In [6]:
# Overview: size, column types, and basic statistics
print(f"{len(df):,} ratings | {df.user.nunique():,} users | {df.item.nunique():,} movies")
df["date"] = pd.to_datetime(df["time"], unit="s")
print("From", df.date.min().date(), "to", df.date.max().date())
df.describe()

1,000,209 ratings | 6,040 users | 3,706 movies
From 2000-04-25 to 2003-02-28


,user,item,rating,time,date
count,1.000209e+06,1.000209e+06,1.000209e+06,1.000209e+06,1000209
mean,3.023512e+03,1.730481e+03,3.581564e+00,9.722437e+08,2000-10-22 19:41:35.404665
min,0.000000e+00,0.000000e+00,1.000000e+00,9.567039e+08,2000-04-25 23:05:32
25%,1.505000e+03,9.650000e+02,3.000000e+00,9.653026e+08,2000-08-03 11:37:17
50%,3.069000e+03,1.657000e+03,4.000000e+00,9.730180e+08,2000-10-31 18:46:46
75%,4.475000e+03,2.565000e+03,4.000000e+00,9.752209e+08,2000-11-26 06:42:19
max,6.039000e+03,3.705000e+03,5.000000e+00,1.046455e+09,2003-02-28 17:49:50
std,1.728413e+03,1.017026e+03,1.117102e+00,1.215256e+07,NaN


In [11]:
# How active are users, how popular are movies, and what does the time split look like?
per_user = df.groupby("user").size()
per_movie = df.groupby("item").size()
train, val, test = split_by_time(df)

pd.DataFrame({
    "ratings per user": per_user.describe(),
    "ratings per movie": per_movie.describe(),
}).round(1), pd.DataFrame({
    "n_events": [len(train), len(val), len(test)],
    "n_users": [train.user.nunique(), val.user.nunique(), test.user.nunique()],
    "first date": [pd.to_datetime(s.time.min(), unit="s").date() for s in (train, val, test)],
    "last date": [pd.to_datetime(s.time.max(), unit="s").date() for s in (train, val, test)],
}, index=["train", "val", "test"])

(       ratings per user  ratings per movie
 count            6040.0             3706.0
 mean              165.6              269.9
 std               192.7              384.0
 min                20.0                1.0
 25%                44.0               33.0
 50%                96.0              123.5
 75%               208.0              350.0
 max              2314.0             3428.0,
        n_events  n_users  first date   last date
 train    800167     5400  2000-04-25  2000-12-02
 val       23841      523  2000-12-02  2000-12-29
 test      80608     1008  2000-12-30  2003-02-28)

# Model